In [1]:
import pandas as pd
import numpy as np 
import joblib
import wandb
import os
import matplotlib.pyplot as plt # gráficos
from sklearn.model_selection import train_test_split # importamos train_test_split para separar los datos
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import cross_validate, train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestRegressor

import time

from utils.funciones_minio import bajar_minio,crear_cliente_minio

In [2]:
cliente = crear_cliente_minio()
ruta_ml = "dataset_ml"
fichero = "embeddings_imagenes.parquet"

In [3]:
df = bajar_minio(cliente,ruta_ml,fichero)

In [4]:
X = np.stack(df['embedding'].values)

y = df['clase'].values.to_numpy()

print("Formato de X:", X.shape)
print("Primeros datos de X:" , X[:10])

print("Formato de Y:", y.shape)
print("Primeros datos de Y:" , y[:10])


Formato de X: (172306, 2048)
Primeros datos de X: [[0.0004778 0.0865    0.        ... 0.0182    0.        0.       ]
 [0.01599   0.2047    0.03983   ... 0.0532    0.01504   0.       ]
 [0.        0.02373   0.00996   ... 0.005074  0.003613  0.       ]
 ...
 [0.        0.05365   0.        ... 0.        0.        0.       ]
 [0.        0.0508    0.1459    ... 0.195     0.05643   0.       ]
 [0.        0.        0.05194   ... 0.0136    0.        0.       ]]
Formato de Y: (172306,)
Primeros datos de Y: ['Cocina' 'Cocina' 'Cocina' 'Cocina' 'Cocina' 'Cocina' 'Cocina' 'Cocina'
 'Cocina' 'Cocina']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101, stratify=y)

print("Tamaño train:", X_train.shape)
print("Tamaño test: ", X_test.shape)

Tamaño train: (120614, 2048)
Tamaño test:  (51692, 2048)


In [8]:
wandb.login(relogin=True)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\harra\_netrc


True

In [16]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="pd1-c2526-team2",
    # Set the wandb project where this run will be logged.
    project="my-awesome-project",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "CNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})


# Finish the run and upload any remaining data.
run.finish()

ServicePollForTokenError: Failed to read port info after 30.0 seconds.

# Baseline (clase predominante)

In [ ]:
run = wandb.init(entity = "pd1-c2526-team2",project="clasificador-imagen",job_type = "train")

modelo_baseline = DummyClassifier(strategy='most_frequent')

modelo_baseline.fit(X_train, y_train)

ruta_mdl = os.path.join(run.dir, "modelo_baseline.joblib")
joblib.dump(modelo_baseline, ruta_mdl)

predicciones = modelo_baseline.predict(X_test)
predicciones_proba = modelo_baseline.predict_proba(X_test)

wandb.sklearn.plot_classifier(
    modelo_baseline, X_train, X_test, y_train, y_test, 
    predicciones, predicciones_proba, 
    labels=['Cocina', 'Dormitorio', 'Salón', 'Banyo'], 
    model_name='Baseline'
)
precision_baseline = accuracy_score(y_test, predicciones)
print(f" Precisión del Baseline: {precision_baseline * 100:.2f}%")

run.finish()



ServicePollForTokenError: Failed to read port info after 30.0 seconds.

In [ ]:
profundidad = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
resultados = []

for p in profundidad:
  arbol = DecisionTreeClassifier(criterion="entropy",max_depth=p,random_state = 101)
  kross_validation = cross_validate(arbol, X_train, y_train, cv=10, scoring='accuracy')
  resultados.append(np.mean(kross_validation["test_score"]))

print(resultados)

In [ ]:
plt.plot(profundidad, resultados)
plt.xlabel('Profundidad')
plt.ylabel('Accuracy')
plt.title('Curva de aprendizaje')
plt.show()

In [ ]:
start = time.time()
tree = DecisionTreeClassifier(criterion="entropy",max_depth=5,random_state = 101)
tree.fit(X_train_norm,y_train)
time_entropy = time.time() - start
print("Tiempo de entrenamiento: ", time_entropy)